# FlashEats - Dependable pipeline checks

This notebook records the reliability contract for the finished one-command pipeline: completeness, validation gates, bounded retries, idempotent outputs, and safe failure. The final entry point is `src/run_pipeline.py`; the older Class 8 implementation is retained under `src/run_pipeline_reference.py` as provenance.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'data' / 'raw' / 'sqlite' / 'flasheats.db').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / 'data' / 'raw'
OUTPUTS = ROOT / 'outputs'
retrieval_runs = sorted((RAW / 'api' / 'retrieval_runs').glob('run_date=*'))
print('final pipeline:', ROOT / 'src' / 'run_pipeline.py')
print('retrieval runs available:', retrieval_runs)
assert (ROOT / 'src' / 'run_pipeline.py').exists()
assert (ROOT / 'src' / 'flasheats_pipeline').is_dir()

## Reliability contract

The final pipeline must publish processed output only after these checks pass or are explicitly marked as warnings.

In [ ]:
reliability_contract = pd.DataFrame([
    ['API completeness', 'reported total equals collected unique order IDs', 'FAIL if false'],
    ['Required schema', 'required columns exist in every source', 'FAIL if false'],
    ['Business grain', 'order_journey has one row per order_id', 'WARN/quarantine duplicates'],
    ['Critical nulls', 'KPI denominator has valid outcome fields', 'WARN and report exclusions'],
    ['Freshness', 'source timestamps meet configured age limit', 'FAIL if stale'],
    ['Retries', 'only transient API failures are retried with a bound', 'FAIL after retry budget'],
    ['Raw preservation', 'successful API pages are written before transformation', 'FAIL if missing'],
    ['Idempotency', 'same run_date replaces the same partition safely', 'FAIL if rows append'],
], columns=['check', 'expectation', 'failure_policy'])
display(reliability_contract)

## Validate a retrieved API run

In [ ]:
if retrieval_runs:
    latest_run = retrieval_runs[-1]
    pages = sorted(latest_run.glob('dispatch_page_*.json'))
    payloads = [json.loads(path.read_text(encoding='utf-8')) for path in pages]
    dispatch_rows = [row for payload in payloads for row in payload.get('data', [])]
    reported_total = payloads[0].get('total_records')
    dispatch = pd.DataFrame(dispatch_rows)
    reliability_result = {
        'run': latest_run.name,
        'pages': len(pages),
        'rows': len(dispatch),
        'unique_order_ids': dispatch.order_id.nunique(),
        'reported_total': reported_total,
        'complete': len(dispatch) == reported_total == dispatch.order_id.nunique(),
    }
    display(pd.DataFrame([reliability_result]))
else:
    print('Run notebook 01 first to create a raw API retrieval run.')

## Idempotent partition demonstration

The final pipeline will use this same logical-partition pattern rather than appending duplicate rows on rerun.

In [ ]:
demo_partition = OUTPUTS / 'reliability_demo' / 'run_date=demo'
demo_partition.mkdir(parents=True, exist_ok=True)
demo_path = demo_partition / 'idempotency_check.csv'
sample = pd.DataFrame({'order_id': ['O00001', 'O00002'], 'run_marker': ['same_run', 'same_run']})
sample.to_csv(demo_path, index=False)
sample.to_csv(demo_path, index=False)
rerun = pd.read_csv(demo_path)
assert len(rerun) == len(sample)
assert rerun.order_id.is_unique
print('Idempotency check passed:', demo_path)

Next implementation step: replace the reference extractor/transformer with the full-source pipeline while preserving this contract and these demonstrations.